In [1]:
import math
import os
import pandas as pd
import numpy as np
from classes import GroundSet, MSDAmazonObjective
from greedy_algorithms import greedy, DP_greedy, DP_sample_greedy, random_baseline
from dp_mechanisms import get_best_eps_0


def run_amazon_experiment(objective, ground_set, params, rep):
    results = []
    k, eps, p, lam, g = params['k'], params['eps'], params['private'], params['lambda'], params['gamma']

    # Sensitivity for Amazon Ratings is usually the max rating (e.g., 5.0) / num_users
    delta_target = 1 / (objective.num_users ** 1.5)
    eps_0 = get_best_eps_0(eps_target=eps, delta_target=delta_target, k=k)

    algorithms = [
        ('nonpriv', greedy, [objective, ground_set, k]),
        ('DPGreedy', DP_greedy, [objective, ground_set, k, eps_0, p]),
        ('DPSampleOblGreedy', DP_sample_greedy, [objective, ground_set, k, eps_0, p, True, g]),
        ('DPSampleGreedy', DP_sample_greedy, [objective, ground_set, k, eps_0, p, False, g]),
        ('Random', random_baseline, [objective, ground_set, k])
    ]

    final_selected = {}
    for i in range(rep):
        print(f"\n--- Repetition {i + 1}/{rep} ---")
        for name, func, args in algorithms:
            print(f"Running {name}...")
            res = func(*args)
            selected, value = res[0], res[1]
            queries = res[2] if len(res) > 2 else 0

            results.append({
                'alg': name, 'k': k, 'lambda_param': lam, 'eps': eps,
                'rep': i, 'value': value, 'queries': queries
            })
            final_selected[name] = selected

    df_results = pd.DataFrame(results)
    output_file = "Amazon_Master_Results.csv"
    df_results.to_csv(output_file, mode='a', index=False, header=not os.path.isfile(output_file))

    return final_selected.get('DPGreedy')


In [7]:


# --- Execution ---
reviews_path = "Health_and_Household_Top10k_Dense.csv"
reviews_df = pd.read_csv(reviews_path, header=None, names=['user_id', 'parent_asin', 'rating', 'timestamp'])
reviews_df['rating'] = reviews_df['rating']/5.0
reviews_df = reviews_df.sample(n=100000)
reviews_df.to_csv('Health_and_Household_example.csv',index=False)

meta_path = "Health_and_Household_Final.csv"
meta_df = pd.read_csv(meta_path, sep='\x1f', low_memory=False)

meta_df = meta_df[meta_df['categories'].apply(lambda x: 'Health Care' in x)].head(100)
meta_df.to_csv('meta_Health_and_Household_example.csv', index=False)
# 3. Pre-process Categories into a Dictionary of Sets {asin: {cat1, cat2}}
# This is crucial for the Jaccard distance logic
print("Preparing category lookup...")
product_categories_dict = (
    meta_df.set_index('parent_asin')['categories']
    .astype(str)
    .str.lower()
    .str.split()
    .apply(set)
    .to_dict()
)

# 4. Define the Ground Set (The actual product IDs available to pick from)
all_asins = list(meta_df['parent_asin'].unique())
g_set = GroundSet(elements=all_asins)

param_grid = [
{'k': 4, 'eps': 0.1, 'lambda': 0.05, 'private': False, 'gamma': 0.1},
{'k': 4, 'eps': 0.1, 'lambda': 0, 'private': False, 'gamma': 0.1},
]

for config in param_grid:
    print(
        f"\n================ CONFIG: k={config['k']}, eps={config['eps']}, lam={config['lambda']} ================")
    
    # Initialize Objective with processed data
    obj = MSDAmazonObjective(
        reviews_df=reviews_df,
        product_categories=product_categories_dict,
        lambda_param=config['lambda'],
        k=config['k'],
        distortion=1.0,
    )
    
    run_amazon_experiment(obj, g_set, config, rep=1)  # Reduced reps for speed

Preparing category lookup...

================ CONFIG: k=4, eps=0.1, lam=0.05 ================

--- Repetition 1/1 ---
Running nonpriv...
Iteration 1: Added B0C1G1BJ2B, Total Value: 0.0032
Iteration 2: Added B09JY3BZBS, Total Value: 0.0119
Iteration 3: Added B0C261DBT5, Total Value: 0.0255
Iteration 4: Added B096NGQHC9, Total Value: 0.0446
Running DPGreedy...
Iteration 1: Added B0C1G1BJ2B, Total Value: 0.0032
Iteration 2: Added B09JY3BZBS, Total Value: 0.0119
Iteration 3: Added B0C261DBT5, Total Value: 0.0255
Iteration 4: Added B096NGQHC9, Total Value: 0.0446
Total Value: 0.0507, Coverage: 0.0128, Diversity: 0.7710
Running DPSampleOblGreedy...
Iteration 1: Added B0C1G1BJ2B, Total Value: 0.0063
Iteration 2: Added B09GCYK34V, Total Value: 0.0139
Iteration 3: Added B09JY3BZBS, Total Value: 0.0295
Iteration 4: Added B0C261DBT5, Total Value: 0.0502
Total Value: 0.0502, Coverage: 0.0137, Diversity: 0.7434
Running DPSampleGreedy...
Iteration 1: Added B0BCT8SLHR, Total Value: 0.0013
Iteration 